In [1]:
import numpy as np
import pandas as pd
import joblib
from tensorflow.keras.models import load_model
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============================================================
# 1. LOAD SAVED TUNED LSTM + SCALERS
# ============================================================

model = load_model("Supply_LSTM.keras")

feature_scaler = joblib.load("Supply_LSTM_X_scaler.pkl")
target_scaler = joblib.load("Supply_LSTM_y_scaler.pkl")

# ============================================================
# 2. LOAD THE COMPLETE SUPPLY DATA
#    This must contain Jan-Mar 2026 actual values
# ============================================================

df = pd.read_csv("supply_dataset.csv")

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date").reset_index(drop=True)

# ============================================================
# 3. SAME 18 FEATURES USED DURING LSTM TRAINING
# ============================================================

features = [
    "Coal_Lag_1",
    "Oil_Gas_Lag_1",
    "Nuclear_Lag_1",
    "Hydro_Lag_1",
    "Solar_Lag_1",
    "Wind_Lag_1",
    "Small_Hydro_Lag_1",
    "Bio_Power_Lag_1",
    "Year",
    "Month",
    "Quarter",
    "Total_Lag_1",
    "Total_Lag_3",
    "Total_Lag_6",
    "Total_Lag_12",
    "Total_Rolling_3",
    "Total_Rolling_6",
    "Total_Rolling_12"
]

# ============================================================
# 4. SELECT JAN-MAR 2026
# ============================================================

future = df[
    (df["Date"] >= "2026-01-01") &
    (df["Date"] <= "2026-03-01")
].copy()

print("Actual 2026 data:")
print(future[["Date", "Total"]])

# ============================================================
# 5. CHECK THAT ALL 3 MONTHS EXIST
# ============================================================

if len(future) != 3:
    raise ValueError(
        f"Expected 3 rows for Jan-Mar 2026, but found {len(future)} rows."
    )

# ============================================================
# 6. CREATE INPUT DATA
# ============================================================

X_future = future[features].values

# Scale using the ALREADY FITTED scaler
X_future_scaled = feature_scaler.transform(X_future)

# ============================================================
# 7. CREATE LSTM SEQUENCE
# ============================================================

LOOKBACK = 3

# Need the previous 3 months before Jan 2026
history = df[df["Date"] < "2026-01-01"].tail(LOOKBACK)

X_history = history[features].values
X_history_scaled = feature_scaler.transform(X_history)

combined = np.vstack([
    X_history_scaled,
    X_future_scaled
])

# ============================================================
# 8. RECURSIVE 3-MONTH FORECAST
# ============================================================

predictions = []

sequence = X_history_scaled.copy()

for i in range(3):

    X_input = sequence[-LOOKBACK:]
    X_input = X_input.reshape(1, LOOKBACK, len(features))

    pred_scaled = model.predict(X_input, verbose=0)

    pred = target_scaler.inverse_transform(
        pred_scaled.reshape(-1, 1)
    )[0, 0]

    predictions.append(pred)

    # Add the known feature row for the next month
    sequence = np.vstack([
        sequence,
        X_future_scaled[i]
    ])

# ============================================================
# 9. ACTUAL VALUES
# ============================================================

actual = future["Total"].values

predictions = np.array(predictions)

# ============================================================
# 10. RESULTS TABLE
# ============================================================

results = pd.DataFrame({
    "Date": future["Date"].values,
    "Actual_Supply_MU": actual,
    "Predicted_Supply_MU": predictions
})

results["Error_MU"] = (
    results["Predicted_Supply_MU"]
    - results["Actual_Supply_MU"]
)

results["Absolute_Error_MU"] = (
    results["Error_MU"].abs()
)

results["APE_%"] = (
    results["Absolute_Error_MU"]
    / results["Actual_Supply_MU"]
) * 100

print("\n========================================")
print("JAN-MAR 2026 SUPPLY FORECAST")
print("========================================")

print(results.to_string(index=False))

# ============================================================
# 11. FINAL METRICS
# ============================================================

mae = mean_absolute_error(actual, predictions)

rmse = np.sqrt(
    mean_squared_error(actual, predictions)
)

mape = np.mean(
    np.abs((actual - predictions) / actual)
) * 100

r2 = r2_score(actual, predictions)

print("\n========================================")
print("2026 OUT-OF-SAMPLE RESULTS")
print("========================================")

print(f"MAE  : {mae:.4f} MU")
print(f"RMSE : {rmse:.4f} MU")
print(f"MAPE : {mape:.4f} %")
print(f"R²   : {r2:.4f}")

print("========================================")

Actual 2026 data:
Empty DataFrame
Columns: [Date, Total]
Index: []


ValueError: Expected 3 rows for Jan-Mar 2026, but found 0 rows.